<p style="text-align:center">
        <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="Skills Network Logo">
</p>


### Analyse search terms on the e-commerce web server


##### In this assignment you will download the search term data set for the e-commerce web server and run analytic queries on it.


In [ ]:
# Install spark

In [1]:
!pip install pyspark findspark

In [ ]:
# Start session

In [2]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Analyze Search Terms") \
    .getOrCreate()

26/09/28 04:57:18 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/28 04:57:21 WARN util.Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [ ]:
# Download The search term dataset from the below url
# https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv

In [3]:
!wget -O searchterms.csv "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv"

--2026-09-28 04:57:59--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104, 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 233457 (228K) [text/csv]
Saving to: ‘searchterms.csv’

searchterms.csv     100%[===================>] 227.99K  --.-KB/s    in 0.004s  

2026-09-28 04:58:00 (49.6 MB/s) - ‘searchterms.csv’ saved [233457/233457]



In [ ]:
# Load the csv into a spark dataframe

In [4]:
df = spark.read.csv(
    "searchterms.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+---+-----+----+--------------+
|day|month|year|    searchterm|
+---+-----+----+--------------+
| 12|   11|2021| mobile 6 inch|
| 12|   11|2021| mobile latest|
| 12|   11|2021|   tablet wifi|
| 12|   11|2021|laptop 14 inch|
| 12|   11|2021|     mobile 5g|
+---+-----+----+--------------+
only showing top 5 rows



In [ ]:
# Print the number of rows and columns


In [5]:
print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

Number of rows: 10000
Number of columns: 4


In [ ]:
# Print the top 5 rows


In [6]:
df.show(5)

+---+-----+----+--------------+
|day|month|year|    searchterm|
+---+-----+----+--------------+
| 12|   11|2021| mobile 6 inch|
| 12|   11|2021| mobile latest|
| 12|   11|2021|   tablet wifi|
| 12|   11|2021|laptop 14 inch|
| 12|   11|2021|     mobile 5g|
+---+-----+----+--------------+
only showing top 5 rows



In [ ]:
# Find out the datatype of the column searchterm?


In [7]:
print("searchterm datatype:", df.schema["searchterm"].dataType)

searchterm datatype: StringType


In [ ]:
# How many times was the term `gaming laptop` searched?


In [8]:
gaming_laptop_count = df.filter(
    df.searchterm == "gaming laptop"
).count()

print("gaming laptop searches:", gaming_laptop_count)

gaming laptop searches: 499


In [9]:
# Create temporary SQL table
df.createOrReplaceTempView("searchterms")

In [ ]:
# Print the top 5 most frequently used search terms?


In [10]:
spark.sql("""
    SELECT searchterm, COUNT(*) AS frequency
    FROM searchterms
    GROUP BY searchterm
    ORDER BY frequency DESC
    LIMIT 5
""").show()

[Stage 9:====================================================>  (190 + 8) / 200]

+-------------+---------+
|   searchterm|frequency|
+-------------+---------+
|mobile 6 inch|     2312|
|    mobile 5g|     2301|
|mobile latest|     1327|
|       laptop|      935|
|  tablet wifi|      896|
+-------------+---------+



In [ ]:
# The pretrained sales forecasting model is available at  the below url
# https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz

In [11]:
!wget -O model.tar.gz "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz"

--2026-09-28 05:03:07--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104, 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1490 (1.5K) [application/x-tar]
Saving to: ‘model.tar.gz’

model.tar.gz        100%[===================>]   1.46K  --.-KB/s    in 0s      

2026-09-28 05:03:08 (7.76 MB/s) - ‘model.tar.gz’ saved [1490/1490]



In [ ]:
# Load the sales forecast model.


In [12]:
!tar -xzf model.tar.gz

In [ ]:
# Using the sales forecast model, predict the sales for the year of 2023.


In [13]:
!find . -maxdepth 2 -type d | grep -i model

./sales_prediction.model
./sales_prediction.model/data
./sales_prediction.model/metadata
